# Single-chain scope

In [4]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

from types import SimpleNamespace

from cli import Cli
from dataset_generator.single_transactions_dataset import SingleTransactionsDataset
from dataset_generator.types import DATASET_TYPE

# Define the absolute path for the root directory for outputs here (and if possible, make sure to have a prepared _dataset folder)
OUTPUTS_ROOT = Path("/home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/outputs/mlpbased-single-eval-comparison-ronin12-omni-nomad-polyn-qubit-harmony-thor123-meter-hypr-xbridge")

# Create the SingleTransactionsDataset if it doesn't exist
dataset = SingleTransactionsDataset(root=str(OUTPUTS_ROOT / "_dataset"))
print(f"Dataset size: {len(dataset)} graphs")

Exporting data from database to CSV files... 
Dataset size: 1152 graphs


In [5]:
import os
import csv
import shutil


def split_single_transactions_dataset(
    source_root: str,
    train_root: str,
    test_root: str,
    test_tx_hashes,
    train_tx_hashes=None,
):
    source_processed = os.path.join(source_root, "processed")
    source_raw = os.path.join(source_root, "raw")

    with open(os.path.join(source_processed, "data_index.csv"), "r") as f:
        rows = list(csv.DictReader(f))

    test_tx_hashes = set(test_tx_hashes)
    missing = test_tx_hashes - {r["tx_hash"] for r in rows}
    if missing:
        raise ValueError(f"{len(missing)} train tx_hashes not found in source dataset: {sorted(missing)[:5]}...")

    test_rows = [r for r in rows if r["tx_hash"] in test_tx_hashes]

    if train_tx_hashes is None:
        train_rows = [r for r in rows if r["tx_hash"] not in test_tx_hashes]
    else:
        train_tx_hashes = set(train_tx_hashes)
        missing = train_tx_hashes - {r["tx_hash"] for r in rows}
        if missing:
            raise ValueError(f"{len(missing)} test tx_hashes not found in source dataset: {sorted(missing)[:5]}...")
        overlap = test_tx_hashes & train_tx_hashes
        if overlap:
            raise ValueError(f"{len(overlap)} tx_hashes appear in both splits: {sorted(overlap)[:5]}...")
        train_rows = [r for r in rows if r["tx_hash"] in train_tx_hashes]

    fieldnames = ["index", "graph_id", "bridge", "tx_hash", "file_name", "offset", "synthetic"]

    for split_root, split_rows in [(train_root, train_rows), (test_root, test_rows)]:
        split_raw = os.path.join(split_root, "raw")
        split_processed = os.path.join(split_root, "processed")
        os.makedirs(split_raw, exist_ok=True)
        os.makedirs(split_processed, exist_ok=True)

        # Copy raw CSVs unchanged so SingleTransactionsDataset.__init__ finds
        # raw_file_names present and doesn't trigger a DB download.
        for raw_file in os.listdir(source_raw):
            shutil.copy2(os.path.join(source_raw, raw_file), os.path.join(split_raw, raw_file))

        # Copy only the .pt files this split actually references (dedup, since
        # each file holds up to 50 graphs and several rows may share one file).
        referenced_files = {r["file_name"] for r in split_rows}
        for file_name in referenced_files:
            shutil.copy2(
                os.path.join(source_processed, file_name),
                os.path.join(split_processed, file_name),
            )

        # metadata.json (node/edge type vocab) is dataset-wide, safe to reuse as-is.
        shutil.copy2(
            os.path.join(source_processed, "metadata.json"),
            os.path.join(split_processed, "metadata.json"),
        )

        # Filtered index; file_name/offset are kept as-is since the .pt files were copied verbatim.
        with open(os.path.join(split_processed, "data_index.csv"), "w", newline="") as f:
            writer = csv.DictWriter(f, fieldnames=fieldnames)
            writer.writeheader()
            for new_idx, r in enumerate(split_rows):
                writer.writerow({**r, "index": new_idx})

    return len(train_rows), len(test_rows)


In [6]:
device = "cpu"  # Change to "cuda" if you have a GPU and want to use it
fixed_args = {
    "device": device,
    "dataset": DATASET_TYPE.SINGLE,
    "new_dataset": None,
    "force_reload": False,
    "num_workers": 0,

    "num_epochs": 100,
    "learning_rate": 0.005,
    "augment_factor": 0,
    "dme_threshold": 0.5,
    "rm_semantic_fusion": True,
    "early_stopping": 15,
    "kfolds": 5,
    "first_layer_channels": 128,
    "hidden_channels": 64,
    "dropout": 0.5,
    "input_drop": 0.0,
    "att_drop": 0.0,
    "n_fp_layers": 2,
    "n_mlp_layers": 2,
    "activation": "relu",
    "pooling": "mean",
    "residual": False,
    "random_seed": 42,
    "test_split": 0,    # Notice that we don't want train/test split on a training subset
}

inference_args = {
    "device": device,  # Change to "cuda" if you have a GPU
    "batch_size": 32,
    "num_workers": 0,
    "force_reload": False,
}

## Leak-free run against XChainWatcher labels

In [7]:
# Full-remove nomad-ronin1
from dataset_generator.single_transactions_dataset import SingleTransactionsDataset

SOURCE_ROOT = str(OUTPUTS_ROOT / "_dataset")  # existing generated dataset root
TRAIN_ROOT = str(OUTPUTS_ROOT / "_dataset_train_xchainwatcher_leakfree")
TEST_ROOT = str(OUTPUTS_ROOT / "_dataset_test_xchainwatcher_leakfree")

source_ds = SingleTransactionsDataset(root=SOURCE_ROOT)

# XChainWatcher evaluation: Bridges ronin1 and nomad for testing, remaining cases for training.
test_bridges = {"ronin1", "nomad"}
test_tx_hashes = [
    entry["tx_hash"] for entry in source_ds.data_index
    if entry["bridge"] in test_bridges
]

n_train, n_test = split_single_transactions_dataset(
    source_root=SOURCE_ROOT,
    train_root=TRAIN_ROOT,
    test_root=TEST_ROOT,
    test_tx_hashes=test_tx_hashes,
)
print(f"train: {n_train} graphs, test: {n_test} graphs")

Exporting data from database to CSV files... 
train: 420 graphs, test: 732 graphs


In [8]:
# Run leak-free training on the training subset
run_dir = Cli.train_model(SimpleNamespace(
    **fixed_args,
    load_data=TRAIN_ROOT,
    tags="mlpbased,comparison,xchainwatcher,leakfree"
))

Early stopping: 15
Exporting data from database to CSV files... 
--test-split 0: using the entire dataset for train/val with 5 folds for cross-validation (no held-out test set).
Normalizing node features using min-max normalization.
Label distribution in train/val pool: 357 normal, 63 anomaly.
Identified 262 differential meta-paths for training.
Fold 1/5, epoch 1/100 — training...
  Fold 1, epoch 1, train loss: 0.5637, val loss: 0.2768, PR-AUC (anomaly): 0.8063, f2-score (anomaly): 0.8784
              precision    recall  f1-score   support

           0       1.00      0.87      0.93        71
           1       0.59      1.00      0.74        13

    accuracy                           0.89        84
   macro avg       0.80      0.94      0.84        84
weighted avg       0.94      0.89      0.90        84

Fold 1/5, epoch 2/100 — training...
  Fold 1, epoch 2, train loss: 0.2649, val loss: 0.1203, PR-AUC (anomaly): 0.9600, f2-score (anomaly): 0.9559
              precision    recall

In [9]:
# Run inference on the test subset using the trained model
Cli.evaluate_model(SimpleNamespace(
    **inference_args,
    load_data=TEST_ROOT,
    checkpoint_data=str(Path(run_dir) / "models"),
    tags="mlpbased,comparison,xchainwatcher,leakfree"
))

Loaded 5 fold checkpoints from '/home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/outputs/mlpbased-single-eval-comparison-ronin12-omni-nomad-polyn-qubit-harmony-thor123-meter-hypr-xbridge/train_epochs100_augment0_lr0.005_nosf_es15_ts0_mlpbased_comparison_xchainwatcher_leakfree__202608110849/models'.
Exporting data from database to CSV files... 
Loaded 732 real graphs from '/home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/outputs/mlpbased-single-eval-comparison-ronin12-omni-nomad-polyn-qubit-harmony-thor123-meter-hypr-xbridge/_dataset_test_xchainwatcher_leakfree'.
Fold 1: accuracy=0.7691 roc_auc=0.5001 pr_auc=0.2273 mcc=0.0000
Fold 2: accuracy=0.7691 roc_auc=0.4903 pr_auc=0.2212 mcc=0.0000
Fold 3: accuracy=0.6489 roc_auc=0.6181 pr_auc=0.2643 mcc=-0.2025
Fold 4: accuracy=0.7691 roc_auc=0.4581 pr_auc=0.2172 mcc=0.0000
Fold 5: accuracy=0.7691 roc_auc=0.4810 pr_auc=0.2251 mcc=0.0000

Evaluation report saved to: /home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/outputs/mlpbas

## 5050 testing against XChainWatcher labels

In [10]:
# 5050 testing (50% normal, 50% anomaly)
from collections import defaultdict

from dataset_generator.single_transactions_dataset import SingleTransactionsDataset

SOURCE_ROOT = str(OUTPUTS_ROOT / "_dataset")  # existing generated dataset root
TRAIN_ROOT = str(OUTPUTS_ROOT / "_dataset_train_xchainwatcher_5050")
TEST_ROOT = str(OUTPUTS_ROOT / "_dataset_test_xchainwatcher_5050")

source_ds = SingleTransactionsDataset(root=SOURCE_ROOT)

# XChainWatcher evaluation: Bridges ronin1 and nomad for testing, remaining cases for training.
test_bridges = {"ronin1", "nomad"}
bridge_to_tx_hashes_normal = defaultdict(list)
bridge_to_tx_hashes_anomaly = defaultdict(list)
for idx, entry in enumerate(source_ds.data_index):
    if entry["bridge"] in test_bridges:
        graph = source_ds[idx]
        if int(graph.y[0]) == 0:
            bridge_to_tx_hashes_normal[entry["bridge"]].append(entry["tx_hash"])
        else:
            bridge_to_tx_hashes_anomaly[entry["bridge"]].append(entry["tx_hash"])

test_tx_hashes = []
for bridge in test_bridges:
    tx_hashes = bridge_to_tx_hashes_normal[bridge]
    test_tx_hashes.extend(tx_hashes[: len(tx_hashes) // 2])
    tx_hashes = bridge_to_tx_hashes_anomaly[bridge]
    test_tx_hashes.extend(tx_hashes[: len(tx_hashes) // 2])

n_train, n_test = split_single_transactions_dataset(
    source_root=SOURCE_ROOT,
    train_root=TRAIN_ROOT,
    test_root=TEST_ROOT,
    test_tx_hashes=test_tx_hashes,
)
print(f"train: {n_train} graphs, test: {n_test} graphs")


Exporting data from database to CSV files... 
train: 787 graphs, test: 365 graphs


In [11]:
# Run 5050 training on the training subset
run_dir = Cli.train_model(SimpleNamespace(
    **fixed_args,
    load_data=TRAIN_ROOT,
    tags="mlpbased,comparison,xchainwatcher,5050"
))

Early stopping: 15
Exporting data from database to CSV files... 
--test-split 0: using the entire dataset for train/val with 5 folds for cross-validation (no held-out test set).
Normalizing node features using min-max normalization.
Label distribution in train/val pool: 639 normal, 148 anomaly.
Identified 278 differential meta-paths for training.
Fold 1/5, epoch 1/100 — training...
  Fold 1, epoch 1, train loss: 0.4172, val loss: 0.4715, PR-AUC (anomaly): 0.7798, f2-score (anomaly): 0.8523
              precision    recall  f1-score   support

           0       1.00      0.80      0.89       128
           1       0.54      1.00      0.70        30

    accuracy                           0.84       158
   macro avg       0.77      0.90      0.79       158
weighted avg       0.91      0.84      0.85       158

Fold 1/5, epoch 2/100 — training...
  Fold 1, epoch 2, train loss: 0.1948, val loss: 0.1896, PR-AUC (anomaly): 0.8163, f2-score (anomaly): 0.9615
              precision    recal

In [12]:
# Run inference on the test subset using the trained model
Cli.evaluate_model(SimpleNamespace(
    **inference_args,
    load_data=TEST_ROOT,
    checkpoint_data=str(Path(run_dir) / "models"),
    tags="mlpbased,comparison,xchainwatcher,5050"
))

Loaded 5 fold checkpoints from '/home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/outputs/mlpbased-single-eval-comparison-ronin12-omni-nomad-polyn-qubit-harmony-thor123-meter-hypr-xbridge/train_epochs100_augment0_lr0.005_nosf_es15_ts0_mlpbased_comparison_xchainwatcher_5050__202608110852/models'.
Exporting data from database to CSV files... 
Loaded 365 real graphs from '/home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/outputs/mlpbased-single-eval-comparison-ronin12-omni-nomad-polyn-qubit-harmony-thor123-meter-hypr-xbridge/_dataset_test_xchainwatcher_5050'.
Fold 1: accuracy=0.9671 roc_auc=0.9918 pr_auc=0.9731 mcc=0.9135
Fold 2: accuracy=0.9671 roc_auc=0.9865 pr_auc=0.9495 mcc=0.9135
Fold 3: accuracy=0.9671 roc_auc=0.9806 pr_auc=0.9228 mcc=0.9135
Fold 4: accuracy=0.9671 roc_auc=0.9969 pr_auc=0.9877 mcc=0.9135
Fold 5: accuracy=0.9671 roc_auc=0.9919 pr_auc=0.9797 mcc=0.9135

Evaluation report saved to: /home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/outputs/mlpbased-single

## Leak-free run against XScope labels

In [13]:
# Full-remove thorchain123-qubit
from dataset_generator.single_transactions_dataset import SingleTransactionsDataset

SOURCE_ROOT = str(OUTPUTS_ROOT / "_dataset")  # existing generated dataset root
TRAIN_ROOT = str(OUTPUTS_ROOT / "_dataset_train_xscope_leakfree")
TEST_ROOT = str(OUTPUTS_ROOT / "_dataset_test_xscope_leakfree")

source_ds = SingleTransactionsDataset(root=SOURCE_ROOT)

# XChainWatcher evaluation: Bridges ronin1 and nomad for testing, remaining cases for training.
test_bridges = {"thorchain", "thorchain2", "thorchain3", "qubit"}
test_tx_hashes = [
    entry["tx_hash"] for entry in source_ds.data_index
    if entry["bridge"] in test_bridges
]

n_train, n_test = split_single_transactions_dataset(
    source_root=SOURCE_ROOT,
    train_root=TRAIN_ROOT,
    test_root=TEST_ROOT,
    test_tx_hashes=test_tx_hashes,
)
print(f"train: {n_train} graphs, test: {n_test} graphs")

Exporting data from database to CSV files... 
train: 1107 graphs, test: 45 graphs


In [14]:
# Run leak-free training on the training subset
run_dir = Cli.train_model(SimpleNamespace(
    **fixed_args,
    load_data=TRAIN_ROOT,
    tags="mlpbased,comparison,xscope,leakfree"
))

Early stopping: 15
Exporting data from database to CSV files... 
--test-split 0: using the entire dataset for train/val with 5 folds for cross-validation (no held-out test set).
Normalizing node features using min-max normalization.
Label distribution in train/val pool: 918 normal, 189 anomaly.
Identified 262 differential meta-paths for training.
Fold 1/5, epoch 1/100 — training...
  Fold 1, epoch 1, train loss: 0.2530, val loss: 0.1312, PR-AUC (anomaly): 0.9746, f2-score (anomaly): 0.9406
              precision    recall  f1-score   support

           0       1.00      0.93      0.97       184
           1       0.76      1.00      0.86        38

    accuracy                           0.95       222
   macro avg       0.88      0.97      0.91       222
weighted avg       0.96      0.95      0.95       222

Fold 1/5, epoch 2/100 — training...
  Fold 1, epoch 2, train loss: 0.1364, val loss: 0.0833, PR-AUC (anomaly): 0.9851, f2-score (anomaly): 0.9548
              precision    recal

In [15]:
# Run inference on the test subset using the trained model
Cli.evaluate_model(SimpleNamespace(
    **inference_args,
    load_data=TEST_ROOT,
    checkpoint_data=str(Path(run_dir) / "models"),
    tags="mlpbased,comparison,xscope,leakfree"
))

Loaded 5 fold checkpoints from '/home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/outputs/mlpbased-single-eval-comparison-ronin12-omni-nomad-polyn-qubit-harmony-thor123-meter-hypr-xbridge/train_epochs100_augment0_lr0.005_nosf_es15_ts0_mlpbased_comparison_xscope_leakfree__202608110857/models'.
Exporting data from database to CSV files... 
Loaded 45 real graphs from '/home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/outputs/mlpbased-single-eval-comparison-ronin12-omni-nomad-polyn-qubit-harmony-thor123-meter-hypr-xbridge/_dataset_test_xscope_leakfree'.
Fold 1: accuracy=0.0444 roc_auc=1.0000 pr_auc=1.0000 mcc=0.0000
Fold 2: accuracy=0.0444 roc_auc=1.0000 pr_auc=1.0000 mcc=0.0000
Fold 3: accuracy=0.0444 roc_auc=1.0000 pr_auc=1.0000 mcc=0.0000
Fold 4: accuracy=0.0444 roc_auc=1.0000 pr_auc=1.0000 mcc=0.0000
Fold 5: accuracy=0.0444 roc_auc=0.6512 pr_auc=0.9814 mcc=0.0000

Evaluation report saved to: /home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/outputs/mlpbased-single-eval-c

## 5050 testing against XScope labels

In [16]:
# 5050 testing (50% normal, 50% anomaly)
from collections import defaultdict

from dataset_generator.single_transactions_dataset import SingleTransactionsDataset

SOURCE_ROOT = str(OUTPUTS_ROOT / "_dataset")  # existing generated dataset root
TRAIN_ROOT = str(OUTPUTS_ROOT / "_dataset_train_xscope_5050")
TEST_ROOT = str(OUTPUTS_ROOT / "_dataset_test_xscope_5050")

source_ds = SingleTransactionsDataset(root=SOURCE_ROOT)

# XScope evaluation: Bridges thorchain, thorchain2, thorchain3, and qubit for testing, remaining cases for training.
test_bridges = {"thorchain", "thorchain2", "thorchain3", "qubit"}
bridge_to_tx_hashes_normal = defaultdict(list)
bridge_to_tx_hashes_anomaly = defaultdict(list)
for idx, entry in enumerate(source_ds.data_index):
    if entry["bridge"] in test_bridges:
        graph = source_ds[idx]
        if int(graph.y[0]) == 0:
            bridge_to_tx_hashes_normal[entry["bridge"]].append(entry["tx_hash"])
        else:
            bridge_to_tx_hashes_anomaly[entry["bridge"]].append(entry["tx_hash"])

test_tx_hashes = []
for bridge in test_bridges:
    tx_hashes = bridge_to_tx_hashes_normal[bridge]
    test_tx_hashes.extend(tx_hashes[: len(tx_hashes) // 2])
    tx_hashes = bridge_to_tx_hashes_anomaly[bridge]
    test_tx_hashes.extend(tx_hashes[: len(tx_hashes) // 2])

n_train, n_test = split_single_transactions_dataset(
    source_root=SOURCE_ROOT,
    train_root=TRAIN_ROOT,
    test_root=TEST_ROOT,
    test_tx_hashes=test_tx_hashes,
)
print(f"train: {n_train} graphs, test: {n_test} graphs")


Exporting data from database to CSV files... 
train: 1130 graphs, test: 22 graphs


In [17]:
# Run 5050 training on the training subset
run_dir = Cli.train_model(SimpleNamespace(
    **fixed_args,
    load_data=TRAIN_ROOT,
    tags="mlpbased,comparison,xscope,5050"
))

Early stopping: 15
Exporting data from database to CSV files... 
--test-split 0: using the entire dataset for train/val with 5 folds for cross-validation (no held-out test set).
Normalizing node features using min-max normalization.
Label distribution in train/val pool: 919 normal, 211 anomaly.
Identified 261 differential meta-paths for training.
Fold 1/5, epoch 1/100 — training...
  Fold 1, epoch 1, train loss: 0.3459, val loss: 0.3863, PR-AUC (anomaly): 0.7959, f2-score (anomaly): 0.9234
              precision    recall  f1-score   support

           0       0.99      0.95      0.97       183
           1       0.82      0.95      0.88        43

    accuracy                           0.95       226
   macro avg       0.90      0.95      0.93       226
weighted avg       0.96      0.95      0.95       226

Fold 1/5, epoch 2/100 — training...
  Fold 1, epoch 2, train loss: 0.1632, val loss: 0.1360, PR-AUC (anomaly): 0.8621, f2-score (anomaly): 0.9598
              precision    recal

In [18]:
# Run inference on the test subset using the trained model
Cli.evaluate_model(SimpleNamespace(
    **inference_args,
    load_data=TEST_ROOT,
    checkpoint_data=str(Path(run_dir) / "models"),
    tags="mlpbased,comparison,xscope,5050"
))

Loaded 5 fold checkpoints from '/home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/outputs/mlpbased-single-eval-comparison-ronin12-omni-nomad-polyn-qubit-harmony-thor123-meter-hypr-xbridge/train_epochs100_augment0_lr0.005_nosf_es15_ts0_mlpbased_comparison_xscope_5050__202608110902/models'.
Exporting data from database to CSV files... 
Loaded 22 real graphs from '/home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/outputs/mlpbased-single-eval-comparison-ronin12-omni-nomad-polyn-qubit-harmony-thor123-meter-hypr-xbridge/_dataset_test_xscope_5050'.
Fold 1: accuracy=1.0000 roc_auc=1.0000 pr_auc=1.0000 mcc=1.0000
Fold 2: accuracy=1.0000 roc_auc=1.0000 pr_auc=1.0000 mcc=1.0000
Fold 3: accuracy=1.0000 roc_auc=1.0000 pr_auc=1.0000 mcc=1.0000
Fold 4: accuracy=1.0000 roc_auc=1.0000 pr_auc=1.0000 mcc=1.0000
Fold 5: accuracy=1.0000 roc_auc=1.0000 pr_auc=1.0000 mcc=1.0000

Evaluation report saved to: /home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/outputs/mlpbased-single-eval-compariso

## Middle-ground testing: test set with rotating incidents

### Incident: Nomad bridge

In [19]:
# Full-remove only nomad
from dataset_generator.single_transactions_dataset import SingleTransactionsDataset

SOURCE_ROOT = str(OUTPUTS_ROOT / "_dataset")  # existing generated dataset root
TRAIN_ROOT = str(OUTPUTS_ROOT / "_dataset_train_nomadonly")
TEST_ROOT = str(OUTPUTS_ROOT / "_dataset_test_nomadonly")

source_ds = SingleTransactionsDataset(root=SOURCE_ROOT)

# XChainWatcher evaluation: Bridges ronin1 and nomad for testing, remaining cases for training.
test_bridges = {"nomad"}
test_tx_hashes = [
    entry["tx_hash"] for entry in source_ds.data_index
    if entry["bridge"] in test_bridges
]

n_train, n_test = split_single_transactions_dataset(
    source_root=SOURCE_ROOT,
    train_root=TRAIN_ROOT,
    test_root=TEST_ROOT,
    test_tx_hashes=test_tx_hashes,
)
print(f"train: {n_train} graphs, test: {n_test} graphs")

Exporting data from database to CSV files... 
train: 422 graphs, test: 730 graphs


In [20]:
# Run 5050 training on the training subset
run_dir = Cli.train_model(SimpleNamespace(
    **fixed_args,
    load_data=TRAIN_ROOT,
    tags="mlpbased,comparison,nomadonly"
))

Early stopping: 15
Exporting data from database to CSV files... 
--test-split 0: using the entire dataset for train/val with 5 folds for cross-validation (no held-out test set).
Normalizing node features using min-max normalization.
Label distribution in train/val pool: 357 normal, 65 anomaly.
Identified 261 differential meta-paths for training.
Fold 1/5, epoch 1/100 — training...
  Fold 1, epoch 1, train loss: 0.6102, val loss: 0.3511, PR-AUC (anomaly): 0.6435, f2-score (anomaly): 0.7500
              precision    recall  f1-score   support

           0       0.98      0.78      0.87        72
           1       0.43      0.92      0.59        13

    accuracy                           0.80        85
   macro avg       0.71      0.85      0.73        85
weighted avg       0.90      0.80      0.82        85

Fold 1/5, epoch 2/100 — training...
  Fold 1, epoch 2, train loss: 0.2828, val loss: 0.2671, PR-AUC (anomaly): 0.9632, f2-score (anomaly): 0.7937
              precision    recall

In [21]:
# Run inference on the test subset using the trained model
Cli.evaluate_model(SimpleNamespace(
    **inference_args,
    load_data=TEST_ROOT,
    checkpoint_data=str(Path(run_dir) / "models"),
    tags="mlpbased,comparison,nomadonly"
))

Loaded 5 fold checkpoints from '/home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/outputs/mlpbased-single-eval-comparison-ronin12-omni-nomad-polyn-qubit-harmony-thor123-meter-hypr-xbridge/train_epochs100_augment0_lr0.005_nosf_es15_ts0_mlpbased_comparison_nomadonly__202608110909/models'.
Exporting data from database to CSV files... 
Loaded 730 real graphs from '/home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/outputs/mlpbased-single-eval-comparison-ronin12-omni-nomad-polyn-qubit-harmony-thor123-meter-hypr-xbridge/_dataset_test_nomadonly'.
Fold 1: accuracy=0.7712 roc_auc=0.6525 pr_auc=0.2836 mcc=0.0000
Fold 2: accuracy=0.7712 roc_auc=0.6346 pr_auc=0.2713 mcc=0.0000
Fold 3: accuracy=0.6411 roc_auc=0.7872 pr_auc=0.6154 mcc=0.3359
Fold 4: accuracy=0.4863 roc_auc=0.4766 pr_auc=0.2102 mcc=-0.3438
Fold 5: accuracy=0.5521 roc_auc=0.6413 pr_auc=0.2748 mcc=-0.2886

Evaluation report saved to: /home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/outputs/mlpbased-single-eval-comparison

### Incident: Ronin #1 bridge

In [22]:
# Full-remove only ronin1
from dataset_generator.single_transactions_dataset import SingleTransactionsDataset

SOURCE_ROOT = str(OUTPUTS_ROOT / "_dataset")  # existing generated dataset root
TRAIN_ROOT = str(OUTPUTS_ROOT / "_dataset_train_ronin1only")
TEST_ROOT = str(OUTPUTS_ROOT / "_dataset_test_ronin1only")

source_ds = SingleTransactionsDataset(root=SOURCE_ROOT)

# XChainWatcher evaluation: Bridges ronin1 and ronin1 for testing, remaining cases for training.
test_bridges = {"ronin1"}
test_tx_hashes = [
    entry["tx_hash"] for entry in source_ds.data_index
    if entry["bridge"] in test_bridges
]

n_train, n_test = split_single_transactions_dataset(
    source_root=SOURCE_ROOT,
    train_root=TRAIN_ROOT,
    test_root=TEST_ROOT,
    test_tx_hashes=test_tx_hashes,
)
print(f"train: {n_train} graphs, test: {n_test} graphs")

Exporting data from database to CSV files... 
train: 1150 graphs, test: 2 graphs


In [23]:
# Run 5050 training on the training subset
run_dir = Cli.train_model(SimpleNamespace(
    **fixed_args,
    load_data=TRAIN_ROOT,
    tags="mlpbased,comparison,ronin1only"
))

Early stopping: 15
Exporting data from database to CSV files... 
--test-split 0: using the entire dataset for train/val with 5 folds for cross-validation (no held-out test set).
Normalizing node features using min-max normalization.
Label distribution in train/val pool: 920 normal, 230 anomaly.
Identified 257 differential meta-paths for training.
Fold 1/5, epoch 1/100 — training...
  Fold 1, epoch 1, train loss: 0.3670, val loss: 0.2306, PR-AUC (anomaly): 0.9064, f2-score (anomaly): 0.9414
              precision    recall  f1-score   support

           0       0.99      0.95      0.97       184
           1       0.82      0.98      0.89        46

    accuracy                           0.95       230
   macro avg       0.91      0.96      0.93       230
weighted avg       0.96      0.95      0.95       230

Fold 1/5, epoch 2/100 — training...
  Fold 1, epoch 2, train loss: 0.1370, val loss: 0.0822, PR-AUC (anomaly): 0.9870, f2-score (anomaly): 0.9623
              precision    recal

In [24]:
# Run inference on the test subset using the trained model
Cli.evaluate_model(SimpleNamespace(
    **inference_args,
    load_data=TEST_ROOT,
    checkpoint_data=str(Path(run_dir) / "models"),
    tags="mlpbased,comparison,ronin1only"
))

Loaded 5 fold checkpoints from '/home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/outputs/mlpbased-single-eval-comparison-ronin12-omni-nomad-polyn-qubit-harmony-thor123-meter-hypr-xbridge/train_epochs100_augment0_lr0.005_nosf_es15_ts0_mlpbased_comparison_ronin1only__202608110911/models'.
Exporting data from database to CSV files... 
Loaded 2 real graphs from '/home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/outputs/mlpbased-single-eval-comparison-ronin12-omni-nomad-polyn-qubit-harmony-thor123-meter-hypr-xbridge/_dataset_test_ronin1only'.
Fold 1: accuracy=0.0000 roc_auc=nan pr_auc=1.0000 mcc=0.0000


/home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/.venv/lib/python3.14/site-packages/sklearn/metrics/_ranking.py:442: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/.venv/lib/python3.14/site-packages/sklearn/metrics/_ranking.py:442: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(


Fold 2: accuracy=0.0000 roc_auc=nan pr_auc=1.0000 mcc=0.0000
Fold 3: accuracy=0.0000 roc_auc=nan pr_auc=1.0000 mcc=0.0000


/home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/.venv/lib/python3.14/site-packages/sklearn/metrics/_ranking.py:442: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/.venv/lib/python3.14/site-packages/sklearn/metrics/_ranking.py:442: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(


Fold 4: accuracy=0.0000 roc_auc=nan pr_auc=1.0000 mcc=0.0000
Fold 5: accuracy=0.0000 roc_auc=nan pr_auc=1.0000 mcc=0.0000

Evaluation report saved to: /home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/outputs/mlpbased-single-eval-comparison-ronin12-omni-nomad-polyn-qubit-harmony-thor123-meter-hypr-xbridge/train_epochs100_augment0_lr0.005_nosf_es15_ts0_mlpbased_comparison_ronin1only__202608110911/inference_mlpbased-comparison-ronin1only_202608110918


/home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/.venv/lib/python3.14/site-packages/sklearn/metrics/_ranking.py:442: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(


### Incident: THORchain #1 bridge

In [25]:
# Full-remove only thorchain1
from dataset_generator.single_transactions_dataset import SingleTransactionsDataset

SOURCE_ROOT = str(OUTPUTS_ROOT / "_dataset")  # existing generated dataset root
TRAIN_ROOT = str(OUTPUTS_ROOT / "_dataset_train_thorchain1only")
TEST_ROOT = str(OUTPUTS_ROOT / "_dataset_test_thorchain1only")

source_ds = SingleTransactionsDataset(root=SOURCE_ROOT)

# XChainWatcher evaluation: Bridges ronin1 and thorchain1 for testing, remaining cases for training.
test_bridges = {"thorchain"}
test_tx_hashes = [
    entry["tx_hash"] for entry in source_ds.data_index
    if entry["bridge"] in test_bridges
]

n_train, n_test = split_single_transactions_dataset(
    source_root=SOURCE_ROOT,
    train_root=TRAIN_ROOT,
    test_root=TEST_ROOT,
    test_tx_hashes=test_tx_hashes,
)
print(f"train: {n_train} graphs, test: {n_test} graphs")

Exporting data from database to CSV files... 
train: 1146 graphs, test: 6 graphs


In [26]:
# Run 5050 training on the training subset
run_dir = Cli.train_model(SimpleNamespace(
    **fixed_args,
    load_data=TRAIN_ROOT,
    tags="mlpbased,comparison,thorchain1only"
))

Early stopping: 15
Exporting data from database to CSV files... 
--test-split 0: using the entire dataset for train/val with 5 folds for cross-validation (no held-out test set).
Normalizing node features using min-max normalization.
Label distribution in train/val pool: 920 normal, 226 anomaly.
Identified 257 differential meta-paths for training.
Fold 1/5, epoch 1/100 — training...
  Fold 1, epoch 1, train loss: 0.3736, val loss: 0.2249, PR-AUC (anomaly): 0.8409, f2-score (anomaly): 0.9336
              precision    recall  f1-score   support

           0       0.99      0.93      0.96       184
           1       0.79      0.98      0.87        46

    accuracy                           0.94       230
   macro avg       0.89      0.96      0.92       230
weighted avg       0.95      0.94      0.95       230

Fold 1/5, epoch 2/100 — training...
  Fold 1, epoch 2, train loss: 0.1293, val loss: 0.1308, PR-AUC (anomaly): 0.9348, f2-score (anomaly): 0.9504
              precision    recal

In [27]:
# Run inference on the test subset using the trained model
Cli.evaluate_model(SimpleNamespace(
    **inference_args,
    load_data=TEST_ROOT,
    checkpoint_data=str(Path(run_dir) / "models"),
    tags="mlpbased,comparison,thorchain1only"
))

Loaded 5 fold checkpoints from '/home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/outputs/mlpbased-single-eval-comparison-ronin12-omni-nomad-polyn-qubit-harmony-thor123-meter-hypr-xbridge/train_epochs100_augment0_lr0.005_nosf_es15_ts0_mlpbased_comparison_thorchain1only__202608110918/models'.
Exporting data from database to CSV files... 
Loaded 6 real graphs from '/home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/outputs/mlpbased-single-eval-comparison-ronin12-omni-nomad-polyn-qubit-harmony-thor123-meter-hypr-xbridge/_dataset_test_thorchain1only'.
Fold 1: accuracy=1.0000 roc_auc=nan pr_auc=1.0000 mcc=0.0000


/home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/.venv/lib/python3.14/site-packages/sklearn/metrics/_classification.py:620: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(
/home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/.venv/lib/python3.14/site-packages/sklearn/metrics/_ranking.py:442: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/.venv/lib/python3.14/site-packages/sklearn/metrics/_classification.py:620: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(
/home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/.venv/lib/python3.14/site-packages/sklearn/metrics/_ranking.py:442: U

Fold 2: accuracy=1.0000 roc_auc=nan pr_auc=1.0000 mcc=0.0000
Fold 3: accuracy=1.0000 roc_auc=nan pr_auc=1.0000 mcc=0.0000


/home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/.venv/lib/python3.14/site-packages/sklearn/metrics/_classification.py:620: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(
/home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/.venv/lib/python3.14/site-packages/sklearn/metrics/_ranking.py:442: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/.venv/lib/python3.14/site-packages/sklearn/metrics/_classification.py:620: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(
/home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/.venv/lib/python3.14/site-packages/sklearn/metrics/_ranking.py:442: U

Fold 4: accuracy=1.0000 roc_auc=nan pr_auc=1.0000 mcc=0.0000
Fold 5: accuracy=1.0000 roc_auc=nan pr_auc=1.0000 mcc=0.0000

Evaluation report saved to: /home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/outputs/mlpbased-single-eval-comparison-ronin12-omni-nomad-polyn-qubit-harmony-thor123-meter-hypr-xbridge/train_epochs100_augment0_lr0.005_nosf_es15_ts0_mlpbased_comparison_thorchain1only__202608110918/inference_mlpbased-comparison-thorchain1only_202608110923


/home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/.venv/lib/python3.14/site-packages/sklearn/metrics/_classification.py:620: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(
/home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/.venv/lib/python3.14/site-packages/sklearn/metrics/_ranking.py:442: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(


### Incident: THORchain #2 bridge

In [28]:
# Full-remove only thorchain2
from dataset_generator.single_transactions_dataset import SingleTransactionsDataset

SOURCE_ROOT = str(OUTPUTS_ROOT / "_dataset")  # existing generated dataset root
TRAIN_ROOT = str(OUTPUTS_ROOT / "_dataset_train_thorchain2only")
TEST_ROOT = str(OUTPUTS_ROOT / "_dataset_test_thorchain2only")

source_ds = SingleTransactionsDataset(root=SOURCE_ROOT)

# XChainWatcher evaluation: Bridges ronin1 and thorchain2 for testing, remaining cases for training.
test_bridges = {"thorchain2"}
test_tx_hashes = [
    entry["tx_hash"] for entry in source_ds.data_index
    if entry["bridge"] in test_bridges
]

n_train, n_test = split_single_transactions_dataset(
    source_root=SOURCE_ROOT,
    train_root=TRAIN_ROOT,
    test_root=TEST_ROOT,
    test_tx_hashes=test_tx_hashes,
)
print(f"train: {n_train} graphs, test: {n_test} graphs")

Exporting data from database to CSV files... 
train: 1137 graphs, test: 15 graphs


In [29]:
# Run 5050 training on the training subset
run_dir = Cli.train_model(SimpleNamespace(
    **fixed_args,
    load_data=TRAIN_ROOT,
    tags="mlpbased,comparison,thorchain2only"
))

Early stopping: 15
Exporting data from database to CSV files... 
--test-split 0: using the entire dataset for train/val with 5 folds for cross-validation (no held-out test set).
Normalizing node features using min-max normalization.
Label distribution in train/val pool: 920 normal, 217 anomaly.
Identified 257 differential meta-paths for training.
Fold 1/5, epoch 1/100 — training...
  Fold 1, epoch 1, train loss: 0.3849, val loss: 0.2038, PR-AUC (anomaly): 0.9388, f2-score (anomaly): 0.9348
              precision    recall  f1-score   support

           0       0.99      0.94      0.97       184
           1       0.80      0.98      0.88        44

    accuracy                           0.95       228
   macro avg       0.90      0.96      0.92       228
weighted avg       0.96      0.95      0.95       228

Fold 1/5, epoch 2/100 — training...
  Fold 1, epoch 2, train loss: 0.1556, val loss: 0.1585, PR-AUC (anomaly): 0.9561, f2-score (anomaly): 0.9471
              precision    recal

In [30]:
# Run inference on the test subset using the trained model
Cli.evaluate_model(SimpleNamespace(
    **inference_args,
    load_data=TEST_ROOT,
    checkpoint_data=str(Path(run_dir) / "models"),
    tags="mlpbased,comparison,thorchain2only"
))

Loaded 5 fold checkpoints from '/home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/outputs/mlpbased-single-eval-comparison-ronin12-omni-nomad-polyn-qubit-harmony-thor123-meter-hypr-xbridge/train_epochs100_augment0_lr0.005_nosf_es15_ts0_mlpbased_comparison_thorchain2only__202608110924/models'.
Exporting data from database to CSV files... 
Loaded 15 real graphs from '/home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/outputs/mlpbased-single-eval-comparison-ronin12-omni-nomad-polyn-qubit-harmony-thor123-meter-hypr-xbridge/_dataset_test_thorchain2only'.


/home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/.venv/lib/python3.14/site-packages/sklearn/metrics/_ranking.py:442: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/.venv/lib/python3.14/site-packages/sklearn/metrics/_ranking.py:442: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(


Fold 1: accuracy=0.0000 roc_auc=nan pr_auc=1.0000 mcc=0.0000
Fold 2: accuracy=0.0000 roc_auc=nan pr_auc=1.0000 mcc=0.0000


/home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/.venv/lib/python3.14/site-packages/sklearn/metrics/_ranking.py:442: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/.venv/lib/python3.14/site-packages/sklearn/metrics/_ranking.py:442: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(


Fold 3: accuracy=0.0000 roc_auc=nan pr_auc=1.0000 mcc=0.0000
Fold 4: accuracy=0.0000 roc_auc=nan pr_auc=1.0000 mcc=0.0000
Fold 5: accuracy=0.0000 roc_auc=nan pr_auc=1.0000 mcc=0.0000

Evaluation report saved to: /home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/outputs/mlpbased-single-eval-comparison-ronin12-omni-nomad-polyn-qubit-harmony-thor123-meter-hypr-xbridge/train_epochs100_augment0_lr0.005_nosf_es15_ts0_mlpbased_comparison_thorchain2only__202608110924/inference_mlpbased-comparison-thorchain2only_202608110929


/home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/.venv/lib/python3.14/site-packages/sklearn/metrics/_ranking.py:442: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(


### Incident: THORchain #3 bridge

In [31]:
# Full-remove only thorchain3
from dataset_generator.single_transactions_dataset import SingleTransactionsDataset

SOURCE_ROOT = str(OUTPUTS_ROOT / "_dataset")  # existing generated dataset root
TRAIN_ROOT = str(OUTPUTS_ROOT / "_dataset_train_thorchain3only")
TEST_ROOT = str(OUTPUTS_ROOT / "_dataset_test_thorchain3only")

source_ds = SingleTransactionsDataset(root=SOURCE_ROOT)

# XChainWatcher evaluation: Bridges ronin1 and thorchain3 for testing, remaining cases for training.
test_bridges = {"thorchain3"}
test_tx_hashes = [
    entry["tx_hash"] for entry in source_ds.data_index
    if entry["bridge"] in test_bridges
]

n_train, n_test = split_single_transactions_dataset(
    source_root=SOURCE_ROOT,
    train_root=TRAIN_ROOT,
    test_root=TEST_ROOT,
    test_tx_hashes=test_tx_hashes,
)
print(f"train: {n_train} graphs, test: {n_test} graphs")

Exporting data from database to CSV files... 
train: 1144 graphs, test: 8 graphs


In [32]:
# Run 5050 training on the training subset
run_dir = Cli.train_model(SimpleNamespace(
    **fixed_args,
    load_data=TRAIN_ROOT,
    tags="mlpbased,comparison,thorchain3only"
))

Early stopping: 15
Exporting data from database to CSV files... 
--test-split 0: using the entire dataset for train/val with 5 folds for cross-validation (no held-out test set).
Normalizing node features using min-max normalization.
Label distribution in train/val pool: 918 normal, 226 anomaly.
Identified 257 differential meta-paths for training.
Fold 1/5, epoch 1/100 — training...
  Fold 1, epoch 1, train loss: 0.3208, val loss: 0.2118, PR-AUC (anomaly): 0.9561, f2-score (anomaly): 0.9494
              precision    recall  f1-score   support

           0       0.99      0.96      0.97       183
           1       0.85      0.98      0.91        46

    accuracy                           0.96       229
   macro avg       0.92      0.97      0.94       229
weighted avg       0.97      0.96      0.96       229

Fold 1/5, epoch 2/100 — training...
  Fold 1, epoch 2, train loss: 0.1413, val loss: 0.1000, PR-AUC (anomaly): 0.9860, f2-score (anomaly): 0.9544
              precision    recal

In [33]:
# Run inference on the test subset using the trained model
Cli.evaluate_model(SimpleNamespace(
    **inference_args,
    load_data=TEST_ROOT,
    checkpoint_data=str(Path(run_dir) / "models"),
    tags="mlpbased,comparison,thorchain3only"
))

Loaded 5 fold checkpoints from '/home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/outputs/mlpbased-single-eval-comparison-ronin12-omni-nomad-polyn-qubit-harmony-thor123-meter-hypr-xbridge/train_epochs100_augment0_lr0.005_nosf_es15_ts0_mlpbased_comparison_thorchain3only__202608110929/models'.
Exporting data from database to CSV files... 
Loaded 8 real graphs from '/home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/outputs/mlpbased-single-eval-comparison-ronin12-omni-nomad-polyn-qubit-harmony-thor123-meter-hypr-xbridge/_dataset_test_thorchain3only'.
Fold 1: accuracy=0.2500 roc_auc=1.0000 pr_auc=1.0000 mcc=0.0000
Fold 2: accuracy=0.2500 roc_auc=1.0000 pr_auc=1.0000 mcc=0.0000
Fold 3: accuracy=0.2500 roc_auc=1.0000 pr_auc=1.0000 mcc=0.0000
Fold 4: accuracy=0.2500 roc_auc=1.0000 pr_auc=1.0000 mcc=0.0000
Fold 5: accuracy=0.2500 roc_auc=1.0000 pr_auc=1.0000 mcc=0.0000

Evaluation report saved to: /home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/outputs/mlpbased-single-eval-comp

### Incident: Qubit bridge

In [34]:
# Full-remove only qubit
from dataset_generator.single_transactions_dataset import SingleTransactionsDataset

SOURCE_ROOT = str(OUTPUTS_ROOT / "_dataset")  # existing generated dataset root
TRAIN_ROOT = str(OUTPUTS_ROOT / "_dataset_train_qubitonly")
TEST_ROOT = str(OUTPUTS_ROOT / "_dataset_test_qubitonly")

source_ds = SingleTransactionsDataset(root=SOURCE_ROOT)

# XChainWatcher evaluation: Bridges ronin1 and qubit for testing, remaining cases for training.
test_bridges = {"qubit"}
test_tx_hashes = [
    entry["tx_hash"] for entry in source_ds.data_index
    if entry["bridge"] in test_bridges
]

n_train, n_test = split_single_transactions_dataset(
    source_root=SOURCE_ROOT,
    train_root=TRAIN_ROOT,
    test_root=TEST_ROOT,
    test_tx_hashes=test_tx_hashes,
)
print(f"train: {n_train} graphs, test: {n_test} graphs")

Exporting data from database to CSV files... 
train: 1136 graphs, test: 16 graphs


In [35]:
# Run 5050 training on the training subset
run_dir = Cli.train_model(SimpleNamespace(
    **fixed_args,
    load_data=TRAIN_ROOT,
    tags="mlpbased,comparison,qubitonly"
))

Early stopping: 15
Exporting data from database to CSV files... 
--test-split 0: using the entire dataset for train/val with 5 folds for cross-validation (no held-out test set).
Normalizing node features using min-max normalization.
Label distribution in train/val pool: 920 normal, 216 anomaly.
Identified 257 differential meta-paths for training.
Fold 1/5, epoch 1/100 — training...
  Fold 1, epoch 1, train loss: 0.3417, val loss: 0.1250, PR-AUC (anomaly): 0.9360, f2-score (anomaly): 0.9607
              precision    recall  f1-score   support

           0       1.00      0.95      0.97       184
           1       0.83      1.00      0.91        44

    accuracy                           0.96       228
   macro avg       0.92      0.98      0.94       228
weighted avg       0.97      0.96      0.96       228

Fold 1/5, epoch 2/100 — training...
  Fold 1, epoch 2, train loss: 0.1339, val loss: 0.0888, PR-AUC (anomaly): 0.9653, f2-score (anomaly): 0.9565
              precision    recal

In [36]:
# Run inference on the test subset using the trained model
Cli.evaluate_model(SimpleNamespace(
    **inference_args,
    load_data=TEST_ROOT,
    checkpoint_data=str(Path(run_dir) / "models"),
    tags="mlpbased,comparison,qubitonly"
))

Loaded 5 fold checkpoints from '/home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/outputs/mlpbased-single-eval-comparison-ronin12-omni-nomad-polyn-qubit-harmony-thor123-meter-hypr-xbridge/train_epochs100_augment0_lr0.005_nosf_es15_ts0_mlpbased_comparison_qubitonly__202608110936/models'.
Exporting data from database to CSV files... 
Loaded 16 real graphs from '/home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/outputs/mlpbased-single-eval-comparison-ronin12-omni-nomad-polyn-qubit-harmony-thor123-meter-hypr-xbridge/_dataset_test_qubitonly'.


/home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/.venv/lib/python3.14/site-packages/sklearn/metrics/_classification.py:620: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(
/home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/.venv/lib/python3.14/site-packages/sklearn/metrics/_ranking.py:442: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/.venv/lib/python3.14/site-packages/sklearn/metrics/_ranking.py:442: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(


Fold 1: accuracy=1.0000 roc_auc=nan pr_auc=1.0000 mcc=0.0000
Fold 2: accuracy=0.0000 roc_auc=nan pr_auc=1.0000 mcc=0.0000


/home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/.venv/lib/python3.14/site-packages/sklearn/metrics/_classification.py:620: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(
/home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/.venv/lib/python3.14/site-packages/sklearn/metrics/_ranking.py:442: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/.venv/lib/python3.14/site-packages/sklearn/metrics/_ranking.py:442: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(


Fold 3: accuracy=1.0000 roc_auc=nan pr_auc=1.0000 mcc=0.0000
Fold 4: accuracy=0.0000 roc_auc=nan pr_auc=1.0000 mcc=0.0000
Fold 5: accuracy=1.0000 roc_auc=nan pr_auc=1.0000 mcc=0.0000

Evaluation report saved to: /home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/outputs/mlpbased-single-eval-comparison-ronin12-omni-nomad-polyn-qubit-harmony-thor123-meter-hypr-xbridge/train_epochs100_augment0_lr0.005_nosf_es15_ts0_mlpbased_comparison_qubitonly__202608110936/inference_mlpbased-comparison-qubitonly_202608110942


/home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/.venv/lib/python3.14/site-packages/sklearn/metrics/_classification.py:620: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(
/home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/.venv/lib/python3.14/site-packages/sklearn/metrics/_ranking.py:442: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(


In [ ]:
# Full-remove only ronin1
from dataset_generator.single_transactions_dataset import SingleTransactionsDataset

SOURCE_ROOT = str(OUTPUTS_ROOT / "_dataset")  # existing generated dataset root
TRAIN_ROOT = str(OUTPUTS_ROOT / "_dataset_train_ronin1only")
TEST_ROOT = str(OUTPUTS_ROOT / "_dataset_test_ronin1only")

source_ds = SingleTransactionsDataset(root=SOURCE_ROOT)

# XChainWatcher evaluation: Bridges ronin1 and nomad for testing, remaining cases for training.
test_bridges = {"ronin1"}
test_tx_hashes = [
    entry["tx_hash"] for entry in source_ds.data_index
    if entry["bridge"] in test_bridges
]

n_train, n_test = split_single_transactions_dataset(
    source_root=SOURCE_ROOT,
    train_root=TRAIN_ROOT,
    test_root=TEST_ROOT,
    test_tx_hashes=test_tx_hashes,
)
print(f"train: {n_train} graphs, test: {n_test} graphs")

Exporting data from database to CSV files... 


Processing single-chain graphs dataset: 100%|██████████| 1152/1152 [00:15<00:00, 76.79graph/s]

train: 1150 graphs, test: 2 graphs


# Cross-chain scope

In [38]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

from types import SimpleNamespace

from cli import Cli
from dataset_generator.cctx_dataset import CrossChainTransactionsDataset
from dataset_generator.types import DATASET_TYPE

# Define the absolute path for the root directory for outputs here (and if possible, make sure to have a prepared _dataset folder)
OUTPUTS_ROOT = Path("/home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/outputs/mlpbased-cctx-eval-comparison-ronin12-omni-nomad-polyn-qubit-harmony-thor123-meter-hypr-xbridge")

# Create the CrossChainTransactionsDataset if it doesn't exist
dataset = CrossChainTransactionsDataset(root=str(OUTPUTS_ROOT / "_dataset"))
print(f"Dataset size: {len(dataset)} graphs")

Exporting data from database to CSV files... 


Processing graphs dataset: 100%|██████████| 1241/1241 [00:21<00:00, 57.48graph/s]

Dataset size: 1241 graphs


In [39]:
import os
import csv
import shutil

# TODO CHANGE NAME
def split_single_transactions_dataset(
    source_root: str,
    train_root: str,
    test_root: str,
    test_tx_hashes,
    train_tx_hashes=None,
):
    source_processed = os.path.join(source_root, "processed")
    source_raw = os.path.join(source_root, "raw")

    with open(os.path.join(source_processed, "data_index.csv"), "r") as f:
        rows = list(csv.DictReader(f))

    test_tx_hashes = set(test_tx_hashes)
    missing = test_tx_hashes - {r["tx_hash"] for r in rows}
    if missing:
        raise ValueError(f"{len(missing)} train tx_hashes not found in source dataset: {sorted(missing)[:5]}...")

    test_rows = [r for r in rows if r["tx_hash"] in test_tx_hashes]

    if train_tx_hashes is None:
        train_rows = [r for r in rows if r["tx_hash"] not in test_tx_hashes]
    else:
        train_tx_hashes = set(train_tx_hashes)
        missing = train_tx_hashes - {r["tx_hash"] for r in rows}
        if missing:
            raise ValueError(f"{len(missing)} test tx_hashes not found in source dataset: {sorted(missing)[:5]}...")
        overlap = test_tx_hashes & train_tx_hashes
        if overlap:
            raise ValueError(f"{len(overlap)} tx_hashes appear in both splits: {sorted(overlap)[:5]}...")
        train_rows = [r for r in rows if r["tx_hash"] in train_tx_hashes]

    fieldnames = ["index", "cctx_graph_id", "tx_hash", "bridge", "file_name", "offset"]

    for split_root, split_rows in [(train_root, train_rows), (test_root, test_rows)]:
        split_raw = os.path.join(split_root, "raw")
        split_processed = os.path.join(split_root, "processed")
        os.makedirs(split_raw, exist_ok=True)
        os.makedirs(split_processed, exist_ok=True)

        # Copy raw CSVs unchanged so SingleTransactionsDataset.__init__ finds
        # raw_file_names present and doesn't trigger a DB download.
        for raw_file in os.listdir(source_raw):
            shutil.copy2(os.path.join(source_raw, raw_file), os.path.join(split_raw, raw_file))

        # Copy only the .pt files this split actually references (dedup, since
        # each file holds up to 50 graphs and several rows may share one file).
        referenced_files = {r["file_name"] for r in split_rows}
        for file_name in referenced_files:
            shutil.copy2(
                os.path.join(source_processed, file_name),
                os.path.join(split_processed, file_name),
            )

        # metadata.json (node/edge type vocab) is dataset-wide, safe to reuse as-is.
        shutil.copy2(
            os.path.join(source_processed, "metadata.json"),
            os.path.join(split_processed, "metadata.json"),
        )

        # Filtered index; file_name/offset are kept as-is since the .pt files were copied verbatim.
        with open(os.path.join(split_processed, "data_index.csv"), "w", newline="") as f:
            writer = csv.DictWriter(f, fieldnames=fieldnames)
            writer.writeheader()
            for new_idx, r in enumerate(split_rows):
                writer.writerow({**r, "index": new_idx})

    return len(train_rows), len(test_rows)


In [40]:
device = "cpu"  # Change to "cuda" if you have a GPU and want to use it
fixed_args = {
    "device": device,
    "dataset": DATASET_TYPE.CCTX,
    "new_dataset": None,
    "force_reload": False,
    "num_workers": 0,

    "num_epochs": 100,
    "learning_rate": 0.005,  # Notice that, for the old approach in cctx, we use the same learning rate compared to single scope
    "augment_factor": 0,
    "dme_threshold": 0.5,
    "rm_semantic_fusion": True,
    "early_stopping": 15,
    "kfolds": 5,
    "first_layer_channels": 128,
    "hidden_channels": 64,
    "dropout": 0.5,
    "input_drop": 0.0,
    "att_drop": 0.0,
    "n_fp_layers": 2,
    "n_mlp_layers": 2,
    "activation": "relu",
    "pooling": "mean",
    "residual": False,
    "random_seed": 42,
    "test_split": 0,    # Notice that we don't want train/test split on a training subset
}

inference_args = {
    "device": device,  # Change to "cuda" if you have a GPU
    "batch_size": 32,
    "num_workers": 0,
    "force_reload": False,
}

## Leak-free run against XChainWatcher labels

In [41]:
# Full-remove nomad-ronin1
from dataset_generator.cctx_dataset import CrossChainTransactionsDataset

SOURCE_ROOT = str(OUTPUTS_ROOT / "_dataset")  # existing generated dataset root
TRAIN_ROOT = str(OUTPUTS_ROOT / "_dataset_train_xchainwatcher_leakfree")
TEST_ROOT = str(OUTPUTS_ROOT / "_dataset_test_xchainwatcher_leakfree")

source_ds = CrossChainTransactionsDataset(root=SOURCE_ROOT)

# XChainWatcher evaluation: Bridges ronin1 and nomad for testing, remaining cases for training.
test_bridges = {"ronin1", "nomad"}
test_tx_hashes = [
    entry["tx_hash"] for entry in source_ds.data_index
    if entry["bridge"] in test_bridges
]

n_train, n_test = split_single_transactions_dataset(
    source_root=SOURCE_ROOT,
    train_root=TRAIN_ROOT,
    test_root=TEST_ROOT,
    test_tx_hashes=test_tx_hashes,
)
print(f"train: {n_train} graphs, test: {n_test} graphs")

train: 1194 graphs, test: 47 graphs


In [42]:
# Run leak-free training on the training subset
run_dir = Cli.train_model(SimpleNamespace(
    **fixed_args,
    load_data=TRAIN_ROOT,
    tags="mlpbased,comparison,xchainwatcher,leakfree"
))

Early stopping: 15
--test-split 0: using the entire dataset for train/val with 5 folds for cross-validation (no held-out test set).
Normalizing node features using min-max normalization.
Label distribution in train/val pool: 1110 normal, 84 anomaly.
Identified 511 differential meta-paths for training.
Fold 1/5, epoch 1/100 — training...
  Fold 1, epoch 1, train loss: 0.6813, val loss: 0.5339, PR-AUC (anomaly): 0.2658, f2-score (anomaly): 0.3876
              precision    recall  f1-score   support

           0       0.96      0.77      0.85       222
           1       0.16      0.59      0.26        17

    accuracy                           0.76       239
   macro avg       0.56      0.68      0.56       239
weighted avg       0.90      0.76      0.81       239

Fold 1/5, epoch 2/100 — training...
  Fold 1, epoch 2, train loss: 0.4406, val loss: 0.2284, PR-AUC (anomaly): 0.8431, f2-score (anomaly): 0.8173
              precision    recall  f1-score   support

           0       1.00

In [43]:
# Run inference on the test subset using the trained model
Cli.evaluate_model(SimpleNamespace(
    **inference_args,
    load_data=TEST_ROOT,
    checkpoint_data=str(Path(run_dir) / "models"),
    tags="mlpbased,comparison,xchainwatcher,leakfree"
))

Loaded 5 fold checkpoints from '/home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/outputs/mlpbased-cctx-eval-comparison-ronin12-omni-nomad-polyn-qubit-harmony-thor123-meter-hypr-xbridge/train_epochs100_augment0_lr0.005_nosf_es15_ts0_mlpbased_comparison_xchainwatcher_leakfree__202608110944/models'.
Loaded 47 real graphs from '/home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/outputs/mlpbased-cctx-eval-comparison-ronin12-omni-nomad-polyn-qubit-harmony-thor123-meter-hypr-xbridge/_dataset_test_xchainwatcher_leakfree'.


/home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/.venv/lib/python3.14/site-packages/sklearn/metrics/_classification.py:620: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(
/home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/.venv/lib/python3.14/site-packages/sklearn/metrics/_ranking.py:442: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/.venv/lib/python3.14/site-packages/sklearn/metrics/_ranking.py:1131: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(


Fold 1: accuracy=1.0000 roc_auc=nan pr_auc=0.0000 mcc=0.0000


/home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/.venv/lib/python3.14/site-packages/sklearn/metrics/_classification.py:620: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(
/home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/.venv/lib/python3.14/site-packages/sklearn/metrics/_ranking.py:442: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/.venv/lib/python3.14/site-packages/sklearn/metrics/_ranking.py:1131: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(


Fold 2: accuracy=1.0000 roc_auc=nan pr_auc=0.0000 mcc=0.0000


/home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/.venv/lib/python3.14/site-packages/sklearn/metrics/_classification.py:620: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(
/home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/.venv/lib/python3.14/site-packages/sklearn/metrics/_ranking.py:442: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/.venv/lib/python3.14/site-packages/sklearn/metrics/_ranking.py:1131: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(


Fold 3: accuracy=1.0000 roc_auc=nan pr_auc=0.0000 mcc=0.0000


/home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/.venv/lib/python3.14/site-packages/sklearn/metrics/_classification.py:620: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(
/home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/.venv/lib/python3.14/site-packages/sklearn/metrics/_ranking.py:442: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/.venv/lib/python3.14/site-packages/sklearn/metrics/_ranking.py:1131: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(


Fold 4: accuracy=1.0000 roc_auc=nan pr_auc=0.0000 mcc=0.0000
Fold 5: accuracy=1.0000 roc_auc=nan pr_auc=0.0000 mcc=0.0000

Evaluation report saved to: /home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/outputs/mlpbased-cctx-eval-comparison-ronin12-omni-nomad-polyn-qubit-harmony-thor123-meter-hypr-xbridge/train_epochs100_augment0_lr0.005_nosf_es15_ts0_mlpbased_comparison_xchainwatcher_leakfree__202608110944/inference_mlpbased-comparison-xchainwatcher-leakfree_202608110957


/home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/.venv/lib/python3.14/site-packages/sklearn/metrics/_classification.py:620: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(
/home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/.venv/lib/python3.14/site-packages/sklearn/metrics/_ranking.py:442: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/.venv/lib/python3.14/site-packages/sklearn/metrics/_ranking.py:1131: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(


## 5050 testing against XChainWatcher labels

In [44]:
# 5050 testing (50% normal, 50% anomaly)
from collections import defaultdict

from dataset_generator.cctx_dataset import CrossChainTransactionsDataset

SOURCE_ROOT = str(OUTPUTS_ROOT / "_dataset")  # existing generated dataset root
TRAIN_ROOT = str(OUTPUTS_ROOT / "_dataset_train_xchainwatcher_5050")
TEST_ROOT = str(OUTPUTS_ROOT / "_dataset_test_xchainwatcher_5050")

source_ds = CrossChainTransactionsDataset(root=SOURCE_ROOT)

# XChainWatcher evaluation: Bridges ronin1 and nomad for testing, remaining cases for training.
test_bridges = {"ronin1", "nomad"}
bridge_to_tx_hashes_normal = defaultdict(list)
bridge_to_tx_hashes_anomaly = defaultdict(list)
for idx, entry in enumerate(source_ds.data_index):
    if entry["bridge"] in test_bridges:
        graph = source_ds[idx]
        if int(graph.y[0]) == 0:
            bridge_to_tx_hashes_normal[entry["bridge"]].append(entry["tx_hash"])
        else:
            bridge_to_tx_hashes_anomaly[entry["bridge"]].append(entry["tx_hash"])

test_tx_hashes = []
for bridge in test_bridges:
    tx_hashes = bridge_to_tx_hashes_normal[bridge]
    test_tx_hashes.extend(tx_hashes[: len(tx_hashes) // 2])
    tx_hashes = bridge_to_tx_hashes_anomaly[bridge]
    test_tx_hashes.extend(tx_hashes[: len(tx_hashes) // 2])

n_train, n_test = split_single_transactions_dataset(
    source_root=SOURCE_ROOT,
    train_root=TRAIN_ROOT,
    test_root=TEST_ROOT,
    test_tx_hashes=test_tx_hashes,
)
print(f"train: {n_train} graphs, test: {n_test} graphs")


train: 1218 graphs, test: 23 graphs


In [45]:
# Run 5050 training on the training subset
run_dir = Cli.train_model(SimpleNamespace(
    **fixed_args,
    load_data=TRAIN_ROOT,
    tags="mlpbased,comparison,xchainwatcher,5050"
))

Early stopping: 15
--test-split 0: using the entire dataset for train/val with 5 folds for cross-validation (no held-out test set).
Normalizing node features using min-max normalization.
Label distribution in train/val pool: 1134 normal, 84 anomaly.
Identified 496 differential meta-paths for training.
Fold 1/5, epoch 1/100 — training...
  Fold 1, epoch 1, train loss: 0.7056, val loss: 0.5155, PR-AUC (anomaly): 0.3457, f2-score (anomaly): 0.5435
              precision    recall  f1-score   support

           0       0.97      0.94      0.95       227
           1       0.42      0.59      0.49        17

    accuracy                           0.91       244
   macro avg       0.69      0.76      0.72       244
weighted avg       0.93      0.91      0.92       244

Fold 1/5, epoch 2/100 — training...
  Fold 1, epoch 2, train loss: 0.4845, val loss: 0.2209, PR-AUC (anomaly): 0.8098, f2-score (anomaly): 0.6044
              precision    recall  f1-score   support

           0       0.97

In [46]:
# Run inference on the test subset using the trained model
Cli.evaluate_model(SimpleNamespace(
    **inference_args,
    load_data=TEST_ROOT,
    checkpoint_data=str(Path(run_dir) / "models"),
    tags="mlpbased,comparison,xchainwatcher,5050"
))

Loaded 5 fold checkpoints from '/home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/outputs/mlpbased-cctx-eval-comparison-ronin12-omni-nomad-polyn-qubit-harmony-thor123-meter-hypr-xbridge/train_epochs100_augment0_lr0.005_nosf_es15_ts0_mlpbased_comparison_xchainwatcher_5050__202608110958/models'.
Loaded 23 real graphs from '/home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/outputs/mlpbased-cctx-eval-comparison-ronin12-omni-nomad-polyn-qubit-harmony-thor123-meter-hypr-xbridge/_dataset_test_xchainwatcher_5050'.


/home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/.venv/lib/python3.14/site-packages/sklearn/metrics/_classification.py:620: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(
/home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/.venv/lib/python3.14/site-packages/sklearn/metrics/_ranking.py:442: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/.venv/lib/python3.14/site-packages/sklearn/metrics/_ranking.py:1131: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(


Fold 1: accuracy=1.0000 roc_auc=nan pr_auc=0.0000 mcc=0.0000


/home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/.venv/lib/python3.14/site-packages/sklearn/metrics/_classification.py:620: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(
/home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/.venv/lib/python3.14/site-packages/sklearn/metrics/_ranking.py:442: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/.venv/lib/python3.14/site-packages/sklearn/metrics/_ranking.py:1131: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(
/home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/.venv/lib/python3.14/site-packages/sklearn/metrics/_classification.py:620: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusio

Fold 2: accuracy=1.0000 roc_auc=nan pr_auc=0.0000 mcc=0.0000
Fold 3: accuracy=1.0000 roc_auc=nan pr_auc=0.0000 mcc=0.0000


/home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/.venv/lib/python3.14/site-packages/sklearn/metrics/_classification.py:620: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(
/home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/.venv/lib/python3.14/site-packages/sklearn/metrics/_ranking.py:442: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/.venv/lib/python3.14/site-packages/sklearn/metrics/_ranking.py:1131: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(


Fold 4: accuracy=1.0000 roc_auc=nan pr_auc=0.0000 mcc=0.0000
Fold 5: accuracy=1.0000 roc_auc=nan pr_auc=0.0000 mcc=0.0000

Evaluation report saved to: /home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/outputs/mlpbased-cctx-eval-comparison-ronin12-omni-nomad-polyn-qubit-harmony-thor123-meter-hypr-xbridge/train_epochs100_augment0_lr0.005_nosf_es15_ts0_mlpbased_comparison_xchainwatcher_5050__202608110958/inference_mlpbased-comparison-xchainwatcher-5050_202608111013


/home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/.venv/lib/python3.14/site-packages/sklearn/metrics/_classification.py:620: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(
/home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/.venv/lib/python3.14/site-packages/sklearn/metrics/_ranking.py:442: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/.venv/lib/python3.14/site-packages/sklearn/metrics/_ranking.py:1131: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(


## Leak-free run against XScope labels

In [47]:
# Full-remove thorchain123-qubit
from dataset_generator.cctx_dataset import CrossChainTransactionsDataset

SOURCE_ROOT = str(OUTPUTS_ROOT / "_dataset")  # existing generated dataset root
TRAIN_ROOT = str(OUTPUTS_ROOT / "_dataset_train_xscope_leakfree")
TEST_ROOT = str(OUTPUTS_ROOT / "_dataset_test_xscope_leakfree")

source_ds = CrossChainTransactionsDataset(root=SOURCE_ROOT)

# XChainWatcher evaluation: Bridges ronin1 and nomad for testing, remaining cases for training.
test_bridges = {"thorchain", "thorchain2", "thorchain3", "qubit"}
test_tx_hashes = [
    entry["tx_hash"] for entry in source_ds.data_index
    if entry["bridge"] in test_bridges
]

n_train, n_test = split_single_transactions_dataset(
    source_root=SOURCE_ROOT,
    train_root=TRAIN_ROOT,
    test_root=TEST_ROOT,
    test_tx_hashes=test_tx_hashes,
)
print(f"train: {n_train} graphs, test: {n_test} graphs")

train: 1206 graphs, test: 35 graphs


In [48]:
# Run leak-free training on the training subset
run_dir = Cli.train_model(SimpleNamespace(
    **fixed_args,
    load_data=TRAIN_ROOT,
    tags="mlpbased,comparison,xscope,leakfree"
))

Early stopping: 15
--test-split 0: using the entire dataset for train/val with 5 folds for cross-validation (no held-out test set).
Normalizing node features using min-max normalization.
Label distribution in train/val pool: 1146 normal, 60 anomaly.
Identified 499 differential meta-paths for training.
Fold 1/5, epoch 1/100 — training...
  Fold 1, epoch 1, train loss: 0.6464, val loss: 0.3146, PR-AUC (anomaly): 0.1826, f2-score (anomaly): 0.5682
              precision    recall  f1-score   support

           0       0.99      0.87      0.93       230
           1       0.25      0.83      0.38        12

    accuracy                           0.87       242
   macro avg       0.62      0.85      0.66       242
weighted avg       0.95      0.87      0.90       242

Fold 1/5, epoch 2/100 — training...
  Fold 1, epoch 2, train loss: 0.4917, val loss: 0.3258, PR-AUC (anomaly): 0.3134, f2-score (anomaly): 0.5263
              precision    recall  f1-score   support

           0       0.99

In [49]:
# Run inference on the test subset using the trained model
Cli.evaluate_model(SimpleNamespace(
    **inference_args,
    load_data=TEST_ROOT,
    checkpoint_data=str(Path(run_dir) / "models"),
    tags="mlpbased,comparison,xscope,leakfree"
))

Loaded 5 fold checkpoints from '/home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/outputs/mlpbased-cctx-eval-comparison-ronin12-omni-nomad-polyn-qubit-harmony-thor123-meter-hypr-xbridge/train_epochs100_augment0_lr0.005_nosf_es15_ts0_mlpbased_comparison_xscope_leakfree__202608111014/models'.
Loaded 35 real graphs from '/home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/outputs/mlpbased-cctx-eval-comparison-ronin12-omni-nomad-polyn-qubit-harmony-thor123-meter-hypr-xbridge/_dataset_test_xscope_leakfree'.
Fold 1: accuracy=0.4000 roc_auc=0.7614 pr_auc=0.8226 mcc=0.2073
Fold 2: accuracy=0.8286 roc_auc=0.7273 pr_auc=0.7481 mcc=0.6030
Fold 3: accuracy=0.3429 roc_auc=0.2765 pr_auc=0.5899 mcc=-0.2438
Fold 4: accuracy=0.2857 roc_auc=0.0795 pr_auc=0.5126 mcc=-0.2513
Fold 5: accuracy=0.2857 roc_auc=0.3333 pr_auc=0.6190 mcc=-0.2513

Evaluation report saved to: /home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/outputs/mlpbased-cctx-eval-comparison-ronin12-omni-nomad-polyn-qubit-harmony-

## 5050 testing against XScope labels

In [50]:
# 5050 testing (50% normal, 50% anomaly)
from collections import defaultdict

from dataset_generator.cctx_dataset import CrossChainTransactionsDataset

SOURCE_ROOT = str(OUTPUTS_ROOT / "_dataset")  # existing generated dataset root
TRAIN_ROOT = str(OUTPUTS_ROOT / "_dataset_train_xscope_5050")
TEST_ROOT = str(OUTPUTS_ROOT / "_dataset_test_xscope_5050")

source_ds = CrossChainTransactionsDataset(root=SOURCE_ROOT)

# XScope evaluation: Bridges thorchain, thorchain2, thorchain3, and qubit for testing, remaining cases for training.
test_bridges = {"thorchain", "thorchain2", "thorchain3", "qubit"}
bridge_to_tx_hashes_normal = defaultdict(list)
bridge_to_tx_hashes_anomaly = defaultdict(list)
for idx, entry in enumerate(source_ds.data_index):
    if entry["bridge"] in test_bridges:
        graph = source_ds[idx]
        if int(graph.y[0]) == 0:
            bridge_to_tx_hashes_normal[entry["bridge"]].append(entry["tx_hash"])
        else:
            bridge_to_tx_hashes_anomaly[entry["bridge"]].append(entry["tx_hash"])

test_tx_hashes = []
for bridge in test_bridges:
    tx_hashes = bridge_to_tx_hashes_normal[bridge]
    test_tx_hashes.extend(tx_hashes[: len(tx_hashes) // 2])
    tx_hashes = bridge_to_tx_hashes_anomaly[bridge]
    test_tx_hashes.extend(tx_hashes[: len(tx_hashes) // 2])

n_train, n_test = split_single_transactions_dataset(
    source_root=SOURCE_ROOT,
    train_root=TRAIN_ROOT,
    test_root=TEST_ROOT,
    test_tx_hashes=test_tx_hashes,
)
print(f"train: {n_train} graphs, test: {n_test} graphs")


train: 1224 graphs, test: 17 graphs


In [51]:
# Run 5050 training on the training subset
run_dir = Cli.train_model(SimpleNamespace(
    **fixed_args,
    load_data=TRAIN_ROOT,
    tags="mlpbased,comparison,xscope,5050"
))

Early stopping: 15
--test-split 0: using the entire dataset for train/val with 5 folds for cross-validation (no held-out test set).
Normalizing node features using min-max normalization.
Label distribution in train/val pool: 1152 normal, 72 anomaly.
Identified 493 differential meta-paths for training.
Fold 1/5, epoch 1/100 — training...
  Fold 1, epoch 1, train loss: 0.5865, val loss: 0.3529, PR-AUC (anomaly): 0.7663, f2-score (anomaly): 0.6250
              precision    recall  f1-score   support

           0       0.98      0.94      0.96       231
           1       0.42      0.71      0.53        14

    accuracy                           0.93       245
   macro avg       0.70      0.83      0.74       245
weighted avg       0.95      0.93      0.94       245

Fold 1/5, epoch 2/100 — training...
  Fold 1, epoch 2, train loss: 0.3953, val loss: 0.2082, PR-AUC (anomaly): 0.8094, f2-score (anomaly): 0.6790
              precision    recall  f1-score   support

           0       0.99

In [52]:
# Run inference on the test subset using the trained model
Cli.evaluate_model(SimpleNamespace(
    **inference_args,
    load_data=TEST_ROOT,
    checkpoint_data=str(Path(run_dir) / "models"),
    tags="mlpbased,comparison,xscope,5050"
))

Loaded 5 fold checkpoints from '/home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/outputs/mlpbased-cctx-eval-comparison-ronin12-omni-nomad-polyn-qubit-harmony-thor123-meter-hypr-xbridge/train_epochs100_augment0_lr0.005_nosf_es15_ts0_mlpbased_comparison_xscope_5050__202608111033/models'.
Loaded 17 real graphs from '/home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/outputs/mlpbased-cctx-eval-comparison-ronin12-omni-nomad-polyn-qubit-harmony-thor123-meter-hypr-xbridge/_dataset_test_xscope_5050'.
Fold 1: accuracy=0.9412 roc_auc=1.0000 pr_auc=1.0000 mcc=0.8593
Fold 2: accuracy=0.9412 roc_auc=0.9833 pr_auc=0.9936 mcc=0.8593
Fold 3: accuracy=0.4706 roc_auc=0.8500 pr_auc=0.9177 mcc=0.1333
Fold 4: accuracy=0.7059 roc_auc=0.8000 pr_auc=0.8349 mcc=0.0000
Fold 5: accuracy=0.7059 roc_auc=0.9833 pr_auc=0.9936 mcc=0.0000

Evaluation report saved to: /home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/outputs/mlpbased-cctx-eval-comparison-ronin12-omni-nomad-polyn-qubit-harmony-thor123-met

## Middle-ground testing: test set with rotating incidents

### Incident: THORchain #2 bridge

In [53]:
# Full-remove only thorchain2
from dataset_generator.cctx_dataset import CrossChainTransactionsDataset

SOURCE_ROOT = str(OUTPUTS_ROOT / "_dataset")  # existing generated dataset root
TRAIN_ROOT = str(OUTPUTS_ROOT / "_dataset_train_thorchain2only")
TEST_ROOT = str(OUTPUTS_ROOT / "_dataset_test_thorchain2only")

source_ds = CrossChainTransactionsDataset(root=SOURCE_ROOT)

# XChainWatcher evaluation: Bridges ronin1 and thorchain2 for testing, remaining cases for training.
test_bridges = {"thorchain2"}
test_tx_hashes = [
    entry["tx_hash"] for entry in source_ds.data_index
    if entry["bridge"] in test_bridges
]

n_train, n_test = split_single_transactions_dataset(
    source_root=SOURCE_ROOT,
    train_root=TRAIN_ROOT,
    test_root=TEST_ROOT,
    test_tx_hashes=test_tx_hashes,
)
print(f"train: {n_train} graphs, test: {n_test} graphs")

train: 1239 graphs, test: 2 graphs


In [54]:
# Run 5050 training on the training subset
run_dir = Cli.train_model(SimpleNamespace(
    **fixed_args,
    load_data=TRAIN_ROOT,
    tags="mlpbased,comparison,thorchain2only"
))

Early stopping: 15
--test-split 0: using the entire dataset for train/val with 5 folds for cross-validation (no held-out test set).
Normalizing node features using min-max normalization.
Label distribution in train/val pool: 1157 normal, 82 anomaly.
Identified 489 differential meta-paths for training.
Fold 1/5, epoch 1/100 — training...
  Fold 1, epoch 1, train loss: 0.6366, val loss: 0.3071, PR-AUC (anomaly): 0.7917, f2-score (anomaly): 0.7895
              precision    recall  f1-score   support

           0       1.00      0.93      0.96       232
           1       0.48      0.94      0.64        16

    accuracy                           0.93       248
   macro avg       0.74      0.93      0.80       248
weighted avg       0.96      0.93      0.94       248

Fold 1/5, epoch 2/100 — training...
  Fold 1, epoch 2, train loss: 0.2908, val loss: 0.1717, PR-AUC (anomaly): 0.7913, f2-score (anomaly): 0.7303
              precision    recall  f1-score   support

           0       0.99

In [55]:
# Run inference on the test subset using the trained model
Cli.evaluate_model(SimpleNamespace(
    **inference_args,
    load_data=TEST_ROOT,
    checkpoint_data=str(Path(run_dir) / "models"),
    tags="mlpbased,comparison,thorchain2only"
))

Loaded 5 fold checkpoints from '/home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/outputs/mlpbased-cctx-eval-comparison-ronin12-omni-nomad-polyn-qubit-harmony-thor123-meter-hypr-xbridge/train_epochs100_augment0_lr0.005_nosf_es15_ts0_mlpbased_comparison_thorchain2only__202608111049/models'.
Loaded 2 real graphs from '/home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/outputs/mlpbased-cctx-eval-comparison-ronin12-omni-nomad-polyn-qubit-harmony-thor123-meter-hypr-xbridge/_dataset_test_thorchain2only'.


/home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/.venv/lib/python3.14/site-packages/sklearn/metrics/_classification.py:620: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(
/home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/.venv/lib/python3.14/site-packages/sklearn/metrics/_ranking.py:442: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/.venv/lib/python3.14/site-packages/sklearn/metrics/_classification.py:620: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(
/home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/.venv/lib/python3.14/site-packages/sklearn/metrics/_ranking.py:442: U

Fold 1: accuracy=1.0000 roc_auc=nan pr_auc=1.0000 mcc=0.0000
Fold 2: accuracy=1.0000 roc_auc=nan pr_auc=1.0000 mcc=0.0000


/home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/.venv/lib/python3.14/site-packages/sklearn/metrics/_classification.py:620: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(
/home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/.venv/lib/python3.14/site-packages/sklearn/metrics/_ranking.py:442: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/.venv/lib/python3.14/site-packages/sklearn/metrics/_ranking.py:442: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(


Fold 3: accuracy=1.0000 roc_auc=nan pr_auc=1.0000 mcc=0.0000
Fold 4: accuracy=0.5000 roc_auc=nan pr_auc=1.0000 mcc=0.0000
Fold 5: accuracy=1.0000 roc_auc=nan pr_auc=1.0000 mcc=0.0000

Evaluation report saved to: /home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/outputs/mlpbased-cctx-eval-comparison-ronin12-omni-nomad-polyn-qubit-harmony-thor123-meter-hypr-xbridge/train_epochs100_augment0_lr0.005_nosf_es15_ts0_mlpbased_comparison_thorchain2only__202608111049/inference_mlpbased-comparison-thorchain2only_202608111105


/home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/.venv/lib/python3.14/site-packages/sklearn/metrics/_classification.py:620: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(
/home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/.venv/lib/python3.14/site-packages/sklearn/metrics/_ranking.py:442: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(


### Incident: THORchain #3 bridge

In [56]:
# Full-remove only thorchain3
from dataset_generator.cctx_dataset import CrossChainTransactionsDataset

SOURCE_ROOT = str(OUTPUTS_ROOT / "_dataset")  # existing generated dataset root
TRAIN_ROOT = str(OUTPUTS_ROOT / "_dataset_train_thorchain3only")
TEST_ROOT = str(OUTPUTS_ROOT / "_dataset_test_thorchain3only")

source_ds = CrossChainTransactionsDataset(root=SOURCE_ROOT)

# XChainWatcher evaluation: Bridges ronin1 and thorchain3 for testing, remaining cases for training.
test_bridges = {"thorchain3"}
test_tx_hashes = [
    entry["tx_hash"] for entry in source_ds.data_index
    if entry["bridge"] in test_bridges
]

n_train, n_test = split_single_transactions_dataset(
    source_root=SOURCE_ROOT,
    train_root=TRAIN_ROOT,
    test_root=TEST_ROOT,
    test_tx_hashes=test_tx_hashes,
)
print(f"train: {n_train} graphs, test: {n_test} graphs")

train: 1235 graphs, test: 6 graphs


In [58]:
# Run 5050 training on the training subset
run_dir = Cli.train_model(SimpleNamespace(
    **fixed_args,
    load_data=TRAIN_ROOT,
    tags="mlpbased,comparison,thorchain3only"
))

Early stopping: 15
--test-split 0: using the entire dataset for train/val with 5 folds for cross-validation (no held-out test set).
Normalizing node features using min-max normalization.
Label distribution in train/val pool: 1157 normal, 78 anomaly.
Identified 488 differential meta-paths for training.
Fold 1/5, epoch 1/100 — training...
  Fold 1, epoch 1, train loss: 0.6558, val loss: 0.4074, PR-AUC (anomaly): 0.7076, f2-score (anomaly): 0.6707
              precision    recall  f1-score   support

           0       0.98      0.95      0.97       232
           1       0.50      0.73      0.59        15

    accuracy                           0.94       247
   macro avg       0.74      0.84      0.78       247
weighted avg       0.95      0.94      0.94       247

Fold 1/5, epoch 2/100 — training...
  Fold 1, epoch 2, train loss: 0.3433, val loss: 0.0667, PR-AUC (anomaly): 0.9405, f2-score (anomaly): 0.9036
              precision    recall  f1-score   support

           0       1.00

In [59]:
# Run inference on the test subset using the trained model
Cli.evaluate_model(SimpleNamespace(
    **inference_args,
    load_data=TEST_ROOT,
    checkpoint_data=str(Path(run_dir) / "models"),
    tags="mlpbased,comparison,thorchain3only"
))

Loaded 5 fold checkpoints from '/home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/outputs/mlpbased-cctx-eval-comparison-ronin12-omni-nomad-polyn-qubit-harmony-thor123-meter-hypr-xbridge/train_epochs100_augment0_lr0.005_nosf_es15_ts0_mlpbased_comparison_thorchain3only__202608111118/models'.
Loaded 6 real graphs from '/home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/outputs/mlpbased-cctx-eval-comparison-ronin12-omni-nomad-polyn-qubit-harmony-thor123-meter-hypr-xbridge/_dataset_test_thorchain3only'.


/home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/.venv/lib/python3.14/site-packages/sklearn/metrics/_classification.py:620: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(
/home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/.venv/lib/python3.14/site-packages/sklearn/metrics/_ranking.py:442: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(


Fold 1: accuracy=1.0000 roc_auc=nan pr_auc=1.0000 mcc=0.0000


/home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/.venv/lib/python3.14/site-packages/sklearn/metrics/_classification.py:620: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(
/home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/.venv/lib/python3.14/site-packages/sklearn/metrics/_ranking.py:442: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(


Fold 2: accuracy=1.0000 roc_auc=nan pr_auc=1.0000 mcc=0.0000


/home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/.venv/lib/python3.14/site-packages/sklearn/metrics/_classification.py:620: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(
/home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/.venv/lib/python3.14/site-packages/sklearn/metrics/_ranking.py:442: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/.venv/lib/python3.14/site-packages/sklearn/metrics/_classification.py:620: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(
/home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/.venv/lib/python3.14/site-packages/sklearn/metrics/_ranking.py:442: U

Fold 3: accuracy=1.0000 roc_auc=nan pr_auc=1.0000 mcc=0.0000
Fold 4: accuracy=1.0000 roc_auc=nan pr_auc=1.0000 mcc=0.0000
Fold 5: accuracy=1.0000 roc_auc=nan pr_auc=1.0000 mcc=0.0000

Evaluation report saved to: /home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/outputs/mlpbased-cctx-eval-comparison-ronin12-omni-nomad-polyn-qubit-harmony-thor123-meter-hypr-xbridge/train_epochs100_augment0_lr0.005_nosf_es15_ts0_mlpbased_comparison_thorchain3only__202608111118/inference_mlpbased-comparison-thorchain3only_202608111132


/home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/.venv/lib/python3.14/site-packages/sklearn/metrics/_classification.py:620: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(
/home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/.venv/lib/python3.14/site-packages/sklearn/metrics/_ranking.py:442: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(


### Incident: Qubit bridge

In [60]:
# Full-remove only qubit
from dataset_generator.cctx_dataset import CrossChainTransactionsDataset

SOURCE_ROOT = str(OUTPUTS_ROOT / "_dataset")  # existing generated dataset root
TRAIN_ROOT = str(OUTPUTS_ROOT / "_dataset_train_qubitonly")
TEST_ROOT = str(OUTPUTS_ROOT / "_dataset_test_qubitonly")

source_ds = CrossChainTransactionsDataset(root=SOURCE_ROOT)

# XChainWatcher evaluation: Bridges ronin1 and qubit for testing, remaining cases for training.
test_bridges = {"qubit"}
test_tx_hashes = [
    entry["tx_hash"] for entry in source_ds.data_index
    if entry["bridge"] in test_bridges
]

n_train, n_test = split_single_transactions_dataset(
    source_root=SOURCE_ROOT,
    train_root=TRAIN_ROOT,
    test_root=TEST_ROOT,
    test_tx_hashes=test_tx_hashes,
)
print(f"train: {n_train} graphs, test: {n_test} graphs")

train: 1214 graphs, test: 27 graphs


In [61]:
# Run 5050 training on the training subset
run_dir = Cli.train_model(SimpleNamespace(
    **fixed_args,
    load_data=TRAIN_ROOT,
    tags="mlpbased,comparison,qubitonly"
))

Early stopping: 15
--test-split 0: using the entire dataset for train/val with 5 folds for cross-validation (no held-out test set).
Normalizing node features using min-max normalization.
Label distribution in train/val pool: 1146 normal, 68 anomaly.
Identified 495 differential meta-paths for training.
Fold 1/5, epoch 1/100 — training...
  Fold 1, epoch 1, train loss: 0.6465, val loss: 0.3258, PR-AUC (anomaly): 0.5973, f2-score (anomaly): 0.5000
              precision    recall  f1-score   support

           0       0.98      0.83      0.90       230
           1       0.21      0.77      0.33        13

    accuracy                           0.83       243
   macro avg       0.60      0.80      0.62       243
weighted avg       0.94      0.83      0.87       243

Fold 1/5, epoch 2/100 — training...
  Fold 1, epoch 2, train loss: 0.3467, val loss: 0.1222, PR-AUC (anomaly): 0.8360, f2-score (anomaly): 0.7534
              precision    recall  f1-score   support

           0       0.99

In [62]:
# Run inference on the test subset using the trained model
Cli.evaluate_model(SimpleNamespace(
    **inference_args,
    load_data=TEST_ROOT,
    checkpoint_data=str(Path(run_dir) / "models"),
    tags="mlpbased,comparison,qubitonly"
))

Loaded 5 fold checkpoints from '/home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/outputs/mlpbased-cctx-eval-comparison-ronin12-omni-nomad-polyn-qubit-harmony-thor123-meter-hypr-xbridge/train_epochs100_augment0_lr0.005_nosf_es15_ts0_mlpbased_comparison_qubitonly__202608111133/models'.
Loaded 27 real graphs from '/home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/outputs/mlpbased-cctx-eval-comparison-ronin12-omni-nomad-polyn-qubit-harmony-thor123-meter-hypr-xbridge/_dataset_test_qubitonly'.
Fold 1: accuracy=0.1852 roc_auc=0.0000 pr_auc=0.4238 mcc=-0.6447
Fold 2: accuracy=0.2963 roc_auc=0.4545 pr_auc=0.5575 mcc=-0.4264
Fold 3: accuracy=0.2963 roc_auc=0.0000 pr_auc=0.4238 mcc=-0.4264
Fold 4: accuracy=0.2963 roc_auc=0.0000 pr_auc=0.4204 mcc=-0.4264
Fold 5: accuracy=0.1852 roc_auc=0.0000 pr_auc=0.4225 mcc=-0.6447

Evaluation report saved to: /home/gonca/Documents/GitHub/IST-Thesis/BridgeDefender/outputs/mlpbased-cctx-eval-comparison-ronin12-omni-nomad-polyn-qubit-harmony-thor123-me